# MOD-03 · NB-01 — Hypothesis Testing for Health Outcomes
### Health Analytics with Python · Module 03: Statistical Inference
---
**Learning objectives**
- Select the right test for continuous, categorical, and count outcomes
- Apply chi-square, Fisher's exact, t-test, and ANOVA in clinical contexts
- Interpret p-values, effect sizes, and confidence intervals correctly
- Correct for multiple comparisons (Bonferroni, Benjamini-Hochberg FDR)
- Distinguish statistical significance from clinical significance

**Estimated time:** 2 hours | **Level:** Intermediate | **Libraries:** `pandas`, `numpy`, `scipy`, `statsmodels`, `pingouin`


## 1. Setup & Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import (chi2_contingency, fisher_exact, ttest_ind,
                          ttest_rel, mannwhitneyu, wilcoxon, kruskal,
                          f_oneway, shapiro, levene, normaltest)
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.power import TTestIndPower
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.02)
plt.rcParams.update({'figure.dpi':120,'figure.facecolor':'white',
                     'axes.spines.top':False,'axes.spines.right':False})

# ── Synthetic clinical dataset ────────────────────────────────────────────────
np.random.seed(42); N = 1000
base = np.random.normal(size=(N,3))
def logistic(x): return 1/(1+np.exp(-x))

ages       = np.random.normal(62,16,N).clip(18,95).astype(int)
sexes      = np.random.choice(['M','F'],N,p=[0.48,0.52])
payers     = np.random.choice(['Medicare','Medicaid','Private','Self-pay','Other'],
                               N,p=[0.40,0.22,0.28,0.07,0.03])
los_days   = np.random.gamma(2.5,1.8,N).clip(1,30).astype(int)
charges    = (los_days*np.random.uniform(1800,4200,N)).round(2)
np.random.seed(99)
diabetes      = np.random.binomial(1,logistic(0.6*base[:,0]-0.5),N)
hypertension  = np.random.binomial(1,logistic(0.7*base[:,0]-0.2),N)
heart_failure = np.random.binomial(1,logistic(0.4*base[:,1]+0.5*hypertension-1.5),N)
readmitted_30 = np.random.binomial(1,logistic(-2.5+0.4*heart_failure+0.3*diabetes+0.2*(los_days>7)),N)
admit_type    = np.random.choice(['Emergency','Elective','Urgent','Newborn'],N,p=[0.52,0.30,0.16,0.02])

np.random.seed(21)
glucose    = np.random.normal(105+15*diabetes,28,N).clip(50,400).round(1)
creatinine = np.random.gamma(1.6+0.8*heart_failure,0.6,N).clip(0.4,10).round(2)
hba1c      = np.random.normal(6.5+0.8*diabetes,1.4,N).clip(4,14).round(1)
sbp        = np.random.normal(128+12*hypertension,18,N).clip(80,200).round(0)

df = pd.DataFrame({
    'patient_id'   :[f'PT-{i:05d}' for i in range(1,N+1)],
    'age':ages,'sex':sexes,'payer':payers,'admit_type':admit_type,
    'los_days':los_days,'total_charge':charges,
    'diabetes':diabetes,'hypertension':hypertension,
    'heart_failure':heart_failure,'readmitted_30':readmitted_30,
    'glucose':glucose,'creatinine':creatinine,'hba1c':hba1c,'sbp':sbp,
})
df['age_group'] = pd.cut(df['age'],[17,44,64,74,95],labels=['18-44','45-64','65-74','75+'])

print(f"Dataset: {df.shape}  |  Readmission rate: {df.readmitted_30.mean()*100:.1f}%")
df.describe(percentiles=[.25,.50,.75]).round(2)


## 2. Test Selection Framework

```
Outcome type?
├── Categorical (proportions) ─────────────────────────────────────────────────
│     ├── 2 groups, expected counts ≥5 → Chi-square test
│     ├── 2 groups, small N or sparse cells → Fisher's exact test
│     └── >2 groups → Chi-square (or log-linear model)
│
├── Continuous, 2 groups ──────────────────────────────────────────────────────
│     ├── Check normality (Shapiro-Wilk n<50, D'Agostino n≥50)
│     │     ├── Normal + equal variance → Independent samples t-test
│     │     ├── Normal + unequal variance → Welch's t-test
│     │     └── Non-normal (LOS, charges) → Mann-Whitney U
│     └── Paired / matched → Paired t-test (normal) or Wilcoxon signed-rank
│
├── Continuous, >2 groups ─────────────────────────────────────────────────────
│     ├── Normal + equal variance → One-way ANOVA + post-hoc Tukey HSD
│     └── Non-normal → Kruskal-Wallis + Dunn's test
│
└── Count / rate outcome → Poisson test or negative binomial
```


## 3. Normality Assessment

In [ ]:
def normality_report(series, name):
    s = series.dropna()
    skewness = stats.skew(s)
    if len(s) < 50:
        stat, p = shapiro(s)
        test_name = "Shapiro-Wilk"
    else:
        stat, p = normaltest(s)   # D'Agostino-Pearson
        test_name = "D'Agostino"
    normal = p > 0.05
    print(f"  {name:20s}: skew={skewness:+.2f} | {test_name} p={p:.4f} "
          f"→ {'NORMAL' if normal else 'NON-NORMAL'}")
    return normal

print("Normality assessment:")
norm_results = {}
for col, label in [('age','Age'),('los_days','LOS'),('total_charge','Charge'),
                    ('glucose','Glucose'),('creatinine','Creatinine'),('sbp','SBP')]:
    norm_results[col] = normality_report(df[col], label)

print()
print("Decision summary:")
for col, is_normal in norm_results.items():
    rec = "→ Use t-test / ANOVA" if is_normal else "→ Use Mann-Whitney / Kruskal-Wallis"
    print(f"  {col:15s}: {rec}")


## 4. Chi-Square & Fisher's Exact Tests (Categorical Outcomes)

In [ ]:
# ── 4.1  Chi-square: sex × readmission ───────────────────────────────────────
ct = pd.crosstab(df['sex'], df['readmitted_30'], margins=True)
print("Contingency table — Sex × 30-day readmission:")
display(ct)

chi2, p, dof, expected = chi2_contingency(pd.crosstab(df['sex'], df['readmitted_30']))
print(f"\nChi-square: χ²={chi2:.3f}, df={dof}, p={p:.4f}")
print(f"Effect size (Cramér V): {np.sqrt(chi2/(len(df)*min(2-1,2-1))):.3f}")
print(f"Minimum expected count: {expected.min():.1f} (>5 = chi-square is appropriate)")


In [ ]:
# ── 4.2  Fisher's exact: small-cell scenario ─────────────────────────────────
# Rare outcome: heart failure among Newborn admissions
rare_ct = pd.crosstab(df['admit_type']=='Newborn', df['heart_failure'])
print("Contingency table — Newborn vs other × Heart failure:")
display(rare_ct)

or_val, p_fisher = fisher_exact(rare_ct)
print(f"\nFisher's exact test: OR={or_val:.3f}, p={p_fisher:.4f}")
print("→ Use Fisher's exact when any expected cell count < 5")


In [ ]:
# ── 4.3  Relative Risk and Odds Ratio ─────────────────────────────────────────
from scipy.stats import norm as norm_dist

def rr_or(exposed, outcome, label=''):
    a = ((exposed==1)&(outcome==1)).sum()   # exposed + outcome
    b = ((exposed==1)&(outcome==0)).sum()   # exposed + no outcome
    c = ((exposed==0)&(outcome==1)).sum()   # unexposed + outcome
    d = ((exposed==0)&(outcome==0)).sum()   # unexposed + no outcome
    n_exp   = a+b; n_unexp = c+d
    risk_e  = a/n_exp; risk_u = c/n_unexp
    RR = risk_e/risk_u
    OR = (a*d)/(b*c)
    # 95% CI (log method)
    se_log_rr = np.sqrt(1/a - 1/n_exp + 1/c - 1/n_unexp)
    se_log_or = np.sqrt(1/a + 1/b + 1/c + 1/d)
    rr_lo,rr_hi = np.exp(np.log(RR)-1.96*se_log_rr), np.exp(np.log(RR)+1.96*se_log_rr)
    or_lo,or_hi = np.exp(np.log(OR)-1.96*se_log_or), np.exp(np.log(OR)+1.96*se_log_or)
    print(f"{label}")
    print(f"  Risk (exposed)  : {risk_e:.3f}  |  Risk (unexposed): {risk_u:.3f}")
    print(f"  RR: {RR:.2f} (95% CI: {rr_lo:.2f}–{rr_hi:.2f})")
    print(f"  OR: {OR:.2f} (95% CI: {or_lo:.2f}–{or_hi:.2f})")
    return RR, OR

# Fix: proper CI calculation
def rr_or_correct(exposed, outcome, label=''):
    a = ((exposed==1)&(outcome==1)).sum()
    b = ((exposed==1)&(outcome==0)).sum()
    c = ((exposed==0)&(outcome==1)).sum()
    d = ((exposed==0)&(outcome==0)).sum()
    for ct,nm in [((a+b,c+d),'N')]:pass
    n_exp,n_unexp = a+b,c+d
    risk_e,risk_u = a/n_exp,c/n_unexp
    RR = risk_e/risk_u; OR = (a*d)/(b*c) if b*c>0 else np.nan
    se_log_rr = np.sqrt((1-risk_e)/(a)+(1-risk_u)/(c))
    se_log_or = np.sqrt(1/a+1/b+1/c+1/d)
    rr_lo,rr_hi = np.exp(np.log(RR)-1.96*se_log_rr),np.exp(np.log(RR)+1.96*se_log_rr)
    or_lo,or_hi = np.exp(np.log(OR)-1.96*se_log_or),np.exp(np.log(OR)+1.96*se_log_or)
    print(f"\n{label}")
    print(f"  RR = {RR:.2f} (95% CI: {rr_lo:.2f}–{rr_hi:.2f})")
    print(f"  OR = {OR:.2f} (95% CI: {or_lo:.2f}–{or_hi:.2f})")

rr_or_correct(df['diabetes'], df['readmitted_30'], "Diabetes → 30-day readmission")
rr_or_correct(df['heart_failure'], df['readmitted_30'], "Heart failure → 30-day readmission")


## 5. Independent Samples & Non-Parametric Tests (Continuous Outcomes)

In [ ]:
# ── 5.1  LOS by readmission status (non-normal → Mann-Whitney U) ─────────────
g0 = df.loc[df['readmitted_30']==0,'los_days']
g1 = df.loc[df['readmitted_30']==1,'los_days']

stat_mw, p_mw = mannwhitneyu(g0,g1,alternative='two-sided')
# Effect size: rank-biserial correlation
r_rb = 1 - (2*stat_mw)/(len(g0)*len(g1))

print("LOS by 30-day readmission status (Mann-Whitney U)")
print(f"  Not readmitted: median = {g0.median():.1f} days [IQR: {g0.quantile(.25):.1f}–{g0.quantile(.75):.1f}]")
print(f"  Readmitted    : median = {g1.median():.1f} days [IQR: {g1.quantile(.25):.1f}–{g1.quantile(.75):.1f}]")
print(f"  U = {stat_mw:.0f}, p = {p_mw:.4f}")
print(f"  Rank-biserial r = {r_rb:.3f}  (|r|>0.1 small, >0.3 medium, >0.5 large)")


In [ ]:
# ── 5.2  SBP by sex (normal → Welch t-test) ──────────────────────────────────
m_sbp = df.loc[df['sex']=='M','sbp']
f_sbp = df.loc[df['sex']=='F','sbp']

# Levene's test for equal variance
lev_stat, lev_p = levene(m_sbp, f_sbp)
equal_var = lev_p > 0.05

t_stat, p_t = ttest_ind(m_sbp, f_sbp, equal_var=equal_var)
cohen_d = (m_sbp.mean()-f_sbp.mean()) / np.sqrt(
    ((len(m_sbp)-1)*m_sbp.std()**2 + (len(f_sbp)-1)*f_sbp.std()**2)/(len(m_sbp)+len(f_sbp)-2))

print(f"SBP by sex — {'Welch' if not equal_var else 'Student'} t-test")
print(f"  Males  : {m_sbp.mean():.1f} ± {m_sbp.std():.1f} mmHg")
print(f"  Females: {f_sbp.mean():.1f} ± {f_sbp.std():.1f} mmHg")
print(f"  Levene p={lev_p:.3f} → {'Unequal variances → Welch' if not equal_var else 'Equal variances → Student'}")
print(f"  t = {t_stat:.3f}, p = {p_t:.4f}")
print(f"  Cohen's d = {cohen_d:.3f}  (|d|<0.2 trivial, <0.5 small, <0.8 medium, ≥0.8 large)")


## 6. One-Way ANOVA & Post-Hoc Tests

In [ ]:
# ── 6.1  ANOVA: LOS across payer groups (Kruskal-Wallis — non-normal) ─────────
groups_los = [df.loc[df['payer']==p,'los_days'].values for p in df['payer'].unique()]
kw_stat, p_kw = kruskal(*groups_los)

print("LOS across payer groups (Kruskal-Wallis test)")
print(f"H = {kw_stat:.3f}, p = {p_kw:.4f}")
print()
print("Median LOS by payer:")
display(df.groupby('payer')['los_days']
          .agg(['median','mean','count'])
          .rename(columns={'median':'Median','mean':'Mean','count':'N'})
          .sort_values('Median',ascending=False).round(1))


In [ ]:
# ── 6.2  ANOVA: Glucose across age groups (normal → one-way ANOVA) ────────────
age_groups = [df.loc[df['age_group']==g,'glucose'].dropna().values
              for g in ['18-44','45-64','65-74','75+']]
f_stat, p_f = f_oneway(*age_groups)
print(f"Glucose by age group — One-way ANOVA: F={f_stat:.3f}, p={p_f:.4f}")

# Post-hoc: Tukey HSD
from statsmodels.stats.multicomp import pairwise_tukeyhsd
tukey = pairwise_tukeyhsd(df['glucose'].dropna(),
                           df.loc[df['glucose'].notna(),'age_group'].astype(str),
                           alpha=0.05)
print("\nTukey HSD post-hoc comparisons:")
print(tukey.summary())


## 7. Multiple Comparisons Correction

In [ ]:
# Test all 10 comorbidities against readmission — naive and corrected
COMORBS = ['diabetes','hypertension','heart_failure','ckd','copd',
           'obesity','afib','depression','anxiety','sleep_apnea']
# Simulate flags for missing comorbidities
np.random.seed(55)
for c in ['ckd','copd','obesity','afib','depression','anxiety','sleep_apnea']:
    if c not in df.columns:
        df[c] = np.random.binomial(1,0.15,N)

raw_p = []
for c in COMORBS:
    ct = pd.crosstab(df[c], df['readmitted_30'])
    _, p, _, _ = chi2_contingency(ct)
    raw_p.append(p)

# Multiple comparisons corrections
reject_bonf, p_bonf, _, _ = multipletests(raw_p, method='bonferroni')
reject_bh,   p_bh,   _, _ = multipletests(raw_p, method='fdr_bh')

mc_df = pd.DataFrame({
    'Comorbidity'     : COMORBS,
    'Raw p'           : raw_p,
    'Sig (α=0.05)'    : [p<0.05 for p in raw_p],
    'Bonferroni p'    : p_bonf,
    'Sig (Bonf)'      : reject_bonf,
    'BH FDR p'        : p_bh,
    'Sig (BH)'        : reject_bh,
}).round(4)
print("Multiple comparisons — 10 simultaneous chi-square tests:")
display(mc_df)
print(f"\nWith α=0.05:")
print(f"  Naive:      {sum(p<0.05 for p in raw_p)} significant")
print(f"  Bonferroni: {reject_bonf.sum()} significant  (most conservative)")
print(f"  BH FDR:     {reject_bh.sum()} significant   (recommended for exploratory analyses)")


## 8. Statistical vs Clinical Significance

In [ ]:
# Power analysis: what N is needed to detect a clinically meaningful difference?
analysis = TTestIndPower()

# Scenario: detecting a 1-day mean difference in LOS (SD ≈ 3.5 days)
effect_d = 1.0 / 3.5   # Cohen's d for 1-day difference

n_needed = analysis.solve_power(effect_size=effect_d, alpha=0.05, power=0.80)
print(f"To detect a 1-day mean LOS difference (SD=3.5 days):")
print(f"  Cohen's d = {effect_d:.3f}")
print(f"  N per group needed (α=0.05, power=0.80): {int(np.ceil(n_needed))}")

# Power curve
fig, axes = plt.subplots(1,2,figsize=(13,5))

# Panel A: Power vs N for different effect sizes
ns = np.arange(20,501,10)
for d, label, color in [(0.2,'d=0.2 (small)','#B0C4DE'),(0.5,'d=0.5 (medium)','#4878CF'),
                         (0.8,'d=0.8 (large)','#1F4E79')]:
    powers = [analysis.solve_power(effect_size=d,nobs1=n,alpha=0.05) for n in ns]
    axes[0].plot(ns,powers,lw=2,label=label,color=color)
axes[0].axhline(0.80,color='red',ls='--',lw=1.2,label='80% power threshold')
axes[0].set_xlabel('N per group'); axes[0].set_ylabel('Statistical power')
axes[0].set_title('Power vs sample size'); axes[0].legend(fontsize=9)

# Panel B: Statistical vs clinical significance illustration
np.random.seed(7)
ns_sim = [50,200,500,2000]
diffs = []; ps = []; ns_out = []
true_diff = 0.5  # clinically trivial difference in LOS (0.5 days)
for n in ns_sim:
    g0 = np.random.normal(5.0, 3.5, n)
    g1 = np.random.normal(5.5, 3.5, n)   # 0.5-day difference (trivial)
    t, p = ttest_ind(g0, g1)
    diffs.append(g1.mean()-g0.mean()); ps.append(p); ns_out.append(n)

colors_sig = ['#D65F5F' if p<0.05 else '#4878CF' for p in ps]
axes[1].bar([str(n) for n in ns_sim], ps, color=colors_sig)
axes[1].axhline(0.05,color='red',ls='--',lw=1.2,label='α = 0.05')
for i,(n,p) in enumerate(zip(ns_sim,ps)):
    axes[1].text(i,p+0.005,f'p={p:.3f}',ha='center',fontsize=9)
axes[1].set_xlabel('Sample size per group')
axes[1].set_ylabel('p-value')
axes[1].set_title('Same 0.5-day LOS difference: p changes with N (clinical significance ≠ statistical significance)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/mod03/power_significance.png',bbox_inches='tight')
plt.show()
print("\nKey insight: With N=2000, a trivial 0.5-day LOS difference becomes 'significant'.")
print("Always report effect sizes alongside p-values.")


In [ ]:
import os; os.makedirs('/tmp/mod03', exist_ok=True)
plt.savefig('/tmp/mod03/power_significance.png',bbox_inches='tight')


## 9. Knowledge Check
1. Why should you use Mann-Whitney U instead of t-test for LOS comparisons in most health datasets?
2. An OR of 2.1 (95% CI: 1.9–2.3) for diabetes → readmission is statistically significant. Is it clinically meaningful? What else do you need to know?
3. You run 20 chi-square tests comparing comorbidities. How many would you expect to be significant at α=0.05 by chance alone?
4. Bonferroni vs BH FDR: when is each preferred in health research?
5. You find p=0.001 for a 0.3-day LOS difference with N=5,000. Should you change clinical practice?


In [ ]:
# Exercise 5 — effect size for the 0.3-day LOS example
g0_ex = np.random.normal(5.0,3.5,5000)
g1_ex = np.random.normal(5.3,3.5,5000)
t_ex,p_ex = ttest_ind(g0_ex,g1_ex)
d_ex = (g1_ex.mean()-g0_ex.mean())/3.5
print(f"p = {p_ex:.4f}, Cohen's d = {d_ex:.3f}")
print(f"Difference = {g1_ex.mean()-g0_ex.mean():.2f} days")
print("Cohen's d < 0.2 = trivial effect, regardless of p-value.")


---
**Next → NB-02: Relative Risk, Odds Ratios & Confidence Intervals**